# TikTok Data Pipeline - Indifferent Dataset
## Zpracování raw dat z agent_data (stance=indifferent)
---

# 1. KNIHOVNY A KONFIGURACE

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Knihovny načteny")

# 2. KONFIGURACE CEST

In [ ]:

RAW_DATA_PATH = r"C:\Users\Michal Tvaroh\PycharmProjects\Jupyter3\facct-raw\agent_data_20260122_074943"


OUTPUT_PATH = r"C:\Users\Michal Tvaroh\PycharmProjects\Jupyter3\data\data_indifferent.csv"

# Ověření existence cesty
if os.path.exists(RAW_DATA_PATH):
    print(f"✓ Raw data path exists: {RAW_DATA_PATH}")
    print(f"  Počet položek: {len(os.listdir(RAW_DATA_PATH))}")
else:
    print(f"✗ CHYBA: Cesta neexistuje: {RAW_DATA_PATH}")

# 3. FUNKCE PRO PARSING NÁZVŮ SLOŽEK

In [ ]:
def parse_user_folder_name(folder_name):
    """
    Parsuje název složky uživatele.
    Formát: 'bertha.giess - beauty - indifferent - 73'
    Vrací: (username, topic, stance, id) nebo None pokud parsing selže
    """
    try:
        parts = folder_name.split(' - ')
        if len(parts) >= 3:
            username = parts[0].strip()
            topic = parts[1].strip()
            stance = parts[2].strip()
            user_id = parts[3].strip() if len(parts) > 3 else None
            return username, topic, stance, user_id
    except:
        pass
    return None


def is_valid_main_folder(folder_name):
    """
    Kontroluje, zda je složka platná _main složka ( zatím teda bez failing predictor).
    """
    return folder_name.endswith('_main') and 'failing' not in folder_name.lower()


# Test parsování
test_name = "bertha.giess - beauty - indifferent - 73"
result = parse_user_folder_name(test_name)
print(f"Test parsing '{test_name}':")
print(f"  → username: {result[0]}, topic: {result[1]}, stance: {result[2]}, id: {result[3]}")

# 4. SKENOVÁNÍ STRUKTURY DAT

In [ ]:
def scan_data_structure(base_path):
    """
    Skenuje strukturu dat a vrací seznam platných cest k interactions.parquet.
    """
    valid_paths = []
    skipped_paths = []
    
    base = Path(base_path)
    
    # Procházení složek uživatelů
    for user_folder in base.iterdir():
        if not user_folder.is_dir():
            continue
        
        # Parsování informací o uživateli
        user_info = parse_user_folder_name(user_folder.name)
        if user_info is None:
            skipped_paths.append((user_folder.name, "Invalid folder name format"))
            continue
        
        username, topic, stance, user_id = user_info
        
        # Hledání _main složky
        for session_folder in user_folder.iterdir():
            if not session_folder.is_dir():
                continue
            
            # Kontrola, zda je to platná _main složka
            if not is_valid_main_folder(session_folder.name):
                if '_main' in session_folder.name:
                    skipped_paths.append((str(session_folder), "Failing predictor"))
                continue
            
            # Hledání interactions.parquet
            interactions_file = session_folder / 'interactions.parquet'
            if interactions_file.exists():
                valid_paths.append({
                    'path': str(interactions_file),
                    'username': username,
                    'topic': topic,
                    'stance': stance,
                    'user_id': user_id,
                    'session_folder': session_folder.name,
                    'run_id': session_folder.name
                })
    
    return valid_paths, skipped_paths


# Skenování
print("Skenuji strukturu dat...")
valid_paths, skipped_paths = scan_data_structure(RAW_DATA_PATH)

print(f"\n✓ Nalezeno {len(valid_paths)} platných sessions")
print(f"  Přeskočeno: {len(skipped_paths)} položek")

# Ukázka nalezených cest
if valid_paths:
    print(f"\nPříklad platné cesty:")
    print(f"  {valid_paths[0]}")

# Ukázka přeskočených
if skipped_paths[:3]:
    print(f"\nPříklady přeskočených:")
    for path, reason in skipped_paths[:3]:
        print(f"  {reason}: {path}")

# 5. STATISTIKY NALEZENÝCH DAT

In [ ]:
# Statistiky podle topic a stance
if valid_paths:
    stats_df = pd.DataFrame(valid_paths)
    
    print("=" * 50)
    print("STATISTIKY NALEZENÝCH DAT")
    print("=" * 50)
    
    print("\n📊 Podle TOPIC:")
    print(stats_df['topic'].value_counts().to_string())
    
    print("\n📊 Podle STANCE:")
    print(stats_df['stance'].value_counts().to_string())
    
    print("\n📊 Kombinace TOPIC × STANCE:")
    print(pd.crosstab(stats_df['topic'], stats_df['stance']))

# 6. NAČTENÍ A SPOJENÍ DAT

In [ ]:
def load_and_merge_data(valid_paths, show_progress=True):
    """
    Načte všechny interactions.parquet soubory a spojí je do jednoho DataFrame.
    """
    all_data = []
    errors = []
    
    iterator = tqdm(valid_paths, desc="Načítání dat") if show_progress else valid_paths
    
    for item in iterator:
        try:
            # Načtení parquet souboru
            df = pd.read_parquet(item['path'])
            
            # Přidání run_id pro identifikaci session
            df['run_id'] = item['run_id']
            
            all_data.append(df)
            
        except Exception as e:
            errors.append((item['path'], str(e)))
    
    if errors:
        print(f"\n⚠️ Chyby při načítání ({len(errors)}):")
        for path, err in errors[:5]:
            print(f"  {path}: {err}")
    
    if all_data:
        # Spojení všech DataFrame
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        return None


# Načtení dat
print("Načítám data ze všech platných sessions...\n")
df_raw = load_and_merge_data(valid_paths)

if df_raw is not None:
    print(f"\n✓ Data načtena!")
    print(f"  Celkem záznamů: {len(df_raw):,}")
    print(f"  Sloupce: {len(df_raw.columns)}")
else:
    print("✗ Nepodařilo se načíst žádná data!")

# 7. NÁHLED NAČTENÝCH DAT

In [ ]:
if df_raw is not None:
    print("SLOUPCE V DATECH:")
    print(list(df_raw.columns))
    print("\nNÁHLED DAT:")
    display(df_raw.head())
    
    print("\nDATOVÉ TYPY:")
    print(df_raw.dtypes)

# 8. PŘIDÁNÍ PREDIKČNÍCH SLOUPCŮ

In [ ]:
def add_prediction_columns(df):
    """
    Přidá predikční sloupce na základě logiky:
    - predicted_topic_match = True pokud video_time_like není NaN (uživatel lajknul)
    - predicted_topic = topic uživatele pokud lajknul, jinak 'random'
    
    Poznámka: predicted_stance_match není relevantní pro indifferent dataset
    """
    df = df.copy()
    
    # predicted_topic_match = True pokud uživatel video lajknul
    df['predicted_topic_match'] = df['video_time_like'].notna()
    
    # predicted_topic = topic uživatele pokud matchuje, jinak 'random'
    df['predicted_topic'] = np.where(
        df['predicted_topic_match'],
        df['topic'],  # Pokud lajknul, video má stejný topic jako uživatel
        'random'      # Pokud nelajknul, video je random/nesouvisející
    )
    
    # predicted_stance_match - není relevantní pro indifferent, nastavíme na False
    df['predicted_stance_match'] = False
    
    return df


# Aplikace
if df_raw is not None:
    df_processed = add_prediction_columns(df_raw)
    
    print("✓ Predikční sloupce přidány")
    print(f"\n📊 Topic Match statistiky:")
    print(f"  Topic Match (lajknutá videa): {df_processed['predicted_topic_match'].sum():,} ({df_processed['predicted_topic_match'].mean()*100:.1f}%)")
    print(f"  Random (nelajknutá videa): {(~df_processed['predicted_topic_match']).sum():,} ({(~df_processed['predicted_topic_match']).mean()*100:.1f}%)")
    
    print(f"\n📊 Predicted Topic distribuce:")
    print(df_processed['predicted_topic'].value_counts())

def add_prediction_columns(df):
    """
    Přidá predikční sloupce na základě logiky:
    - predicted_topic_match = True pokud video_time_like není NaN (uživatel lajknul)
    - predicted_topic = topic uživatele pokud lajknul, jinak 'random'

    Poznámka: predicted_stance_match není relevantní pro indifferent dataset
    """
    df = df.copy()

    # predicted_topic_match = True pokud uživatel video lajknul
    df['predicted_topic_match'] = df['video_time_like'].notna()

    # predicted_topic = topic uživatele pokud matchuje, jinak 'random'
    df['predicted_topic'] = np.where(
        df['predicted_topic_match'],
        df['topic'],  # Pokud lajknul, video má stejný topic jako uživatel
        'random'      # Pokud nelajknul, video je random/nesouvisející
    )

    # predicted_stance_match - není relevantní pro indifferent, nastavíme na False
    df['predicted_stance_match'] = False

    return df


# Aplikace
df_processed = None  # Inicializace

if df_raw is not None:
    df_processed = add_prediction_columns(df_raw)

    print("✓ Predikční sloupce přidány")
    print(f"\n📊 Topic Match statistiky:")
    print(f"  Topic Match (lajknutá videa): {df_processed['predicted_topic_match'].sum():,} ({df_processed['predicted_topic_match'].mean()*100:.1f}%)")
    print(f"  Random (nelajknutá videa): {(~df_processed['predicted_topic_match']).sum():,} ({(~df_processed['predicted_topic_match']).mean()*100:.1f}%)")

    print(f"\n📊 Predicted Topic distribuce:")
    print(df_processed['predicted_topic'].value_counts())
else:
    print("✗ df_raw není k dispozici - spusť nejdřív buňku 6")

# 9. VALIDACE DAT

In [ ]:
def validate_data(df):
    """
    Validuje data a zobrazí statistiky.
    """
    print("=" * 60)
    print("VALIDACE DAT")
    print("=" * 60)
    
    # Základní statistiky
    print(f"\n📊 ZÁKLADNÍ STATISTIKY:")
    print(f"  Celkem záznamů: {len(df):,}")
    print(f"  Unikátních uživatelů (email): {df['user_email'].nunique()}")
    print(f"  Unikátních sessions (run_id): {df['run_id'].nunique()}")
    
    # Topics
    print(f"\n📊 TOPICS ({df['topic'].nunique()}):")
    print(df['topic'].value_counts().to_string())
    
    # Stances
    print(f"\n📊 STANCES:")
    print(df['stance'].value_counts().to_string())
    
    # Akce
    print(f"\n📊 AKCE:")
    print(f"  Skip:     {df['video_action_skip'].sum():,} ({df['video_action_skip'].mean()*100:.1f}%)")
    print(f"  Watch:    {df['video_action_watch'].sum():,} ({df['video_action_watch'].mean()*100:.1f}%)")
    print(f"  Like:     {df['video_action_like'].sum():,} ({df['video_action_like'].mean()*100:.1f}%)")
    print(f"  Bookmark: {df['video_action_bookmark'].sum():,} ({df['video_action_bookmark'].mean()*100:.1f}%)")
    
    # Predikce
    print(f"\n📊 PREDIKCE:")
    print(f"  Topic Match: {df['predicted_topic_match'].sum():,} ({df['predicted_topic_match'].mean()*100:.1f}%)")
    
    # Kontrola konzistence: lajk = topic match
    like_count = df['video_action_like'].sum()
    topic_match_count = df['predicted_topic_match'].sum()
    if like_count == topic_match_count:
        print(f"\n✓ Konzistence OK: Like count = Topic Match count ({like_count:,})")
    else:
        print(f"\n⚠️ Nekonzistence: Like={like_count:,}, Topic Match={topic_match_count:,}")
    
    return True


# Validace
if df_processed is not None:
    validate_data(df_processed)

# 10. FINÁLNÍ NÁHLED

In [ ]:
if df_processed is not None:
    print("FINÁLNÍ STRUKTURA DAT:")
    print("=" * 60)
    
    # Zobrazit relevantní sloupce
    display_cols = [
        'interaction_number', 'video_author', 'video_action_skip', 'video_action_watch',
        'video_action_like', 'user_email', 'topic', 'stance', 
        'predicted_topic_match', 'predicted_topic', 'run_id'
    ]
    display_cols = [c for c in display_cols if c in df_processed.columns]
    
    display(df_processed[display_cols].head(10))
    
    print("\n📊 ROZDĚLENÍ PODLE TOPIC:")
    print(pd.crosstab(df_processed['topic'], df_processed['predicted_topic_match'], margins=True))

# 11. EXPORT DO CSV

In [ ]:
if df_processed is not None:
    # Vytvoření výstupní složky pokud neexistuje
    output_dir = os.path.dirname(OUTPUT_PATH)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"✓ Vytvořena složka: {output_dir}")
    
    # Export
    df_processed.to_csv(OUTPUT_PATH, index=False)
    
    # Ověření
    file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)  # MB
    
    print("\n" + "=" * 60)
    print("✓ DATA ÚSPĚŠNĚ EXPORTOVÁNA!")
    print("=" * 60)
    print(f"  Soubor: {OUTPUT_PATH}")
    print(f"  Velikost: {file_size:.2f} MB")
    print(f"  Záznamů: {len(df_processed):,}")
    print(f"  Sloupců: {len(df_processed.columns)}")
else:
    print("✗ Nelze exportovat - žádná data!")

# 12. OVĚŘENÍ EXPORTU

In [ ]:
# Načtení a ověření exportovaného CSV
if os.path.exists(OUTPUT_PATH):
    df_verify = pd.read_csv(OUTPUT_PATH)
    
    print("OVĚŘENÍ EXPORTOVANÉHO SOUBORU:")
    print(f"  Načteno záznamů: {len(df_verify):,}")
    print(f"  Sloupce: {list(df_verify.columns)}")
    
    # Kontrola klíčových sloupců
    required = ['interaction_number', 'video_action_skip', 'video_action_watch',
                'video_action_like', 'user_email', 'topic', 'stance',
                'run_id', 'predicted_topic_match', 'predicted_topic']
    
    missing = [col for col in required if col not in df_verify.columns]
    if missing:
        print(f"\n⚠️ Chybí sloupce: {missing}")
    else:
        print("\n✓ Všechny potřebné sloupce jsou přítomny!")
        print("\n→ Můžeš spustit tiktok_analysis_indifferent.ipynb")

---
# HOTOVO!

Data jsou připravena v CSV souboru. Spusť analytický notebook `tiktok_analysis_indifferent.ipynb`.

In [ ]:
# KOMPLETNÍ NAČTENÍ DAT
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

RAW_DATA_PATH = r"C:\Users\Michal Tvaroh\PycharmProjects\Jupyter3\facct-raw\agent_data_20260122_074943"
OUTPUT_PATH = r"C:\Users\Michal Tvaroh\PycharmProjects\Jupyter3\data\data_indifferent.csv"

# 1. SKEN SLOŽEK
print("1. Skenuji složky...")
valid_paths = []

for user_folder in os.listdir(RAW_DATA_PATH):
    user_path = os.path.join(RAW_DATA_PATH, user_folder)

    if not os.path.isdir(user_path):
        continue

    parts = user_folder.split(' - ')
    if len(parts) < 3:
        continue

    username, topic, stance = parts[0], parts[1], parts[2]

    for session_folder in os.listdir(user_path):
        session_path = os.path.join(user_path, session_folder)

        if not os.path.isdir(session_path):
            continue

        if session_folder.endswith('_main') and 'failing' not in session_folder.lower():
            interactions_file = os.path.join(session_path, 'interactions.parquet')
            if os.path.exists(interactions_file):
                valid_paths.append({
                    'path': interactions_file,
                    'username': username,
                    'topic': topic,
                    'stance': stance,
                    'run_id': session_folder
                })

print(f"   Nalezeno {len(valid_paths)} platných sessions")

# 2. NAČTENÍ PARQUET SOUBORŮ
print("\n2. Loading parquet")
all_data = []
errors = []

for item in tqdm(valid_paths):
    try:
        df = pd.read_parquet(item['path'])
        df['run_id'] = item['run_id']
        all_data.append(df)
    except Exception as e:
        errors.append((item['path'], str(e)))

if errors:
    print(f"   Chyby: {len(errors)}")
    for path, err in errors[:3]:
        print(f"     {err}")

# 3. SPOJENÍ DAT
print("\n3. Spojuji data...")
df_raw = pd.concat(all_data, ignore_index=True)
print(f"   Celkem řádků: {len(df_raw):,}")

# 4. PŘIDÁNÍ PREDIKČNÍCH SLOUPCŮ
print("\n4. Přidávám predikční sloupce...")
df_processed = df_raw.copy()
df_processed['predicted_topic_match'] = df_processed['video_time_like'].notna()
df_processed['predicted_topic'] = np.where(
    df_processed['predicted_topic_match'],
    df_processed['topic'],
    'random'
)
df_processed['predicted_stance_match'] = False

# 5. EXPORT
print("\n5. Export do CSV...")
df_processed.to_csv(OUTPUT_PATH, index=False)


if os.path.exists(OUTPUT_PATH):
    file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"\n{'='*60}")
    print(f"✓ Done")
    print(f"{'='*60}")
    print(f"  Soubor: {OUTPUT_PATH}")
    print(f"  Velikost: {file_size:.2f} MB")
    print(f"  Záznamů: {len(df_processed):,}")
    print(f"  uživatelů: {df_processed['user_email'].nunique()}")
    print(f"  Topics: {df_processed['topic'].unique().tolist()}")
    print(f"  Topic Match Rate: {df_processed['predicted_topic_match'].mean()*100:.1f}%")